Importar librerias y cargar API

In [32]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI
import ollama
import time

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

MODEL = "gpt-4o-mini"

print("Client ready")

Client ready


1. Prueba de chat básico y temperatura en la respuesta

In [ ]:
#prompt = "Invent a useless product nobody would ever need. Explain why it exists."
prompt = "Describe the world's strangest restaurant."

temperatures = [0.1, 1.5]

for temp in temperatures:

    print("\n==============================")
    print(f"Temperature: {temp}")
    print("==============================\n")

    response = ollama.chat(
        model="tinyllama:1.1b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={
            "temperature": temp,
            "num_predict": 80
        }
    )

    print(response["message"]["content"])


Temperature: 0.1

The world's strangest restaurant is located in a remote, uninhabited island off the coast of the United States. It's called "The Restaurant at the End of the World," and it's been around for over a century.

The restaurant has no windows or doors, and its walls are made entirely of glass. The ceiling is also made of

Temperature: 1.5

Surrounded by a lively and vibrant crowd, there was something about the restaurant that gave it an ethereal air of otherworldliness. The ambiance was electric with excitement and anticipation, the air a mingling mixture of chicken wings, shrimp rolls, and vegetable spring rolls sizzling away on griddles set into the ground


2. Chat en streaming - El modelo genera tokens progresivamente en vez de esperar a tener todo el output completo.

In [14]:
prompt = "Explain what an LLM is in simple terms."

stream = client.responses.create(
    model=MODEL,
    input=prompt,
    stream=True
)

for event in stream:

    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

print()

An LLM, or Large Language Model, is a type of artificial intelligence that can understand and generate human language. Think of it like a very smart robot that has read a vast amount of text from books, websites, and other sources. Because it has learned from this text, it can answer questions, write stories, have conversations, and much more, all by predicting what words should come next based on the patterns it has seen. It's like a super-powered text assistant that can help with a wide range of language-related tasks!


3. Prueba usando "system messages"

In [ ]:
question = "Explain artificial intelligence."

# PROFESSOR STYLE

print("University Professor------------------------")

response_1 = client.responses.create(
    model=MODEL,
    instructions="You are a university professor explaining concepts formally.",
    input=question,
    max_output_tokens=50
)

print(response_1.output_text)

# CHILD STYLE

print("Explain to a Child----------------------")

response_2 = client.responses.create(
    model=MODEL,
    instructions="Explain concepts like you are talking to a 10-year-old child.",
    input=question,
    max_output_tokens=50
)

print(response_2.output_text)


University Professor

Artificial intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. This field encompasses a variety of techniques and methodologies designed to enable machines to perform tasks that typically require human intelligence. These tasks include, but are not limited

Explain to a Child

Okay! Imagine you have a super-smart robot friend. This robot can listen, learn, and solve problems just like we do, but it uses a lot of information and special rules to help it think.

Artificial intelligence, or AI for short, is


4. Prueba usando tools

In [18]:
import json

text = """
Hello team,

Please contact john.doe@company.com for technical support.
You can also reach maria.garcia@gmail.com for marketing questions.

For urgent matters:
support@openai.com

Thank you.
"""

# -----------------------------------
# TOOL DEFINITION
# -----------------------------------

tools = [
    {
        "type": "function",
        "name": "extract_contact_information",
        "description": "Extract emails and identify their purpose from a text.",
        "parameters": {
            "type": "object",
            "properties": {

                "contacts": {
                    "type": "array",
                    "description": "List of extracted contacts",
                    "items": {
                        "type": "object",
                        "properties": {

                            "email": {
                                "type": "string",
                                "description": "Detected email address"
                            },

                            "purpose": {
                                "type": "string",
                                "description": "Reason or department associated with the email"
                            }

                        },
                        "required": ["email", "purpose"]
                    }
                },

                "total_emails": {
                    "type": "integer",
                    "description": "Total number of emails found"
                }

            },
            "required": ["contacts", "total_emails"]
        }
    }
]

# -----------------------------------
# MODEL CALL
# -----------------------------------

response = client.responses.create(
    model=MODEL,
    input=f"""
Extract all contact information from this text.
For each email, identify its purpose.

Text:
{text}
""",
    tools=tools
)

# -----------------------------------
# PRINT TOOL OUTPUT
# -----------------------------------

tool_call = response.output[0]

arguments = json.loads(tool_call.arguments)

print("\n==============================")
print("Extracted Contact Information")
print("==============================\n")

for contact in arguments["contacts"]:
    print(f"Email: {contact['email']}")
    print(f"Purpose: {contact['purpose']}")
    print()

print(f"Total emails found: {arguments['total_emails']}")


Extracted Contact Information

Email: john.doe@company.com
Purpose: technical support

Email: maria.garcia@gmail.com
Purpose: marketing questions

Email: support@openai.com
Purpose: urgent matters

Total emails found: 3


In [33]:
# -----------------------------------
# LOAD API KEY
# -----------------------------------

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# -----------------------------------
# TEST CONFIG
# -----------------------------------

prompt = """
Explain what Retrieval-Augmented Generation (RAG) is,
how embeddings work,
and why vector databases are useful.
Keep the answer concise.
"""

# OpenAI models
openai_models = [
    "gpt-5-nano",
    "gpt-4.1-nano",
    "gpt-4o-mini"
]

# Ollama models
ollama_models = [
    "tinyllama:1.1b",
    "llama3.2:3b"
]

# -----------------------------------
# OPENAI BENCHMARK
# -----------------------------------

print("\n========================================")
print("OPENAI MODEL PERFORMANCE")
print("========================================\n")

for model in openai_models:

    print(f"\nTesting model: {model}")
    print("-" * 40)

    start = time.perf_counter()

    response = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=120
    )

    end = time.perf_counter()

    latency = end - start

    output = response.output_text

    characters = len(output)
    words = len(output.split())

    chars_per_second = characters / latency
    words_per_second = words / latency

    print(f"Latency: {latency:.2f} seconds")
    print(f"Characters/sec: {chars_per_second:.2f}")

    print("\nResponse Preview:\n")
    print(output[:300])

# -----------------------------------
# OLLAMA BENCHMARK
# -----------------------------------

print("\n\n========================================")
print("OLLAMA MODEL PERFORMANCE")
print("========================================\n")

for model in ollama_models:

    print(f"\nTesting model: {model}")
    print("-" * 40)

    start = time.perf_counter()

    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "num_predict": 120
        }
    )

    end = time.perf_counter()

    latency = end - start

    output = response["message"]["content"]

    characters = len(output)
    words = len(output.split())

    chars_per_second = characters / latency
    words_per_second = words / latency

    print(f"Latency: {latency:.2f} seconds")
    print(f"Characters/sec: {chars_per_second:.2f}")

    print("\nResponse Preview:\n")
    print(output[:300])


OPENAI MODEL PERFORMANCE


Testing model: gpt-5-nano
----------------------------------------
Latency: 1.53 seconds
Characters/sec: 0.00

Response Preview:



Testing model: gpt-4.1-nano
----------------------------------------
Latency: 1.87 seconds
Characters/sec: 279.81

Response Preview:

Retrieval-Augmented Generation (RAG) combines a language model with an external knowledge base by retrieving relevant documents or data using embeddings and vector databases. Embeddings are dense vector representations of text or data that capture semantic meaning, enabling comparison and retrieval 

Testing model: gpt-4o-mini
----------------------------------------
Latency: 2.45 seconds
Characters/sec: 288.27

Response Preview:

**Retrieval-Augmented Generation (RAG)** is a model architecture that combines information retrieval and text generation. It retrieves relevant documents from a knowledge base or corpus and uses those documents to enhance the generation of responses. This leads to more i